# SketchyGAN Notebook (Single Category: `balloon_sketch`)

This notebook implements the main pipeline from **SketchyGAN (CVPR 2018)** and adapts it for one category.

### Included paper components
- **HED-style edge augmentation** from photos to sketch-like maps
- **MRU (Masked Residual Unit)** blocks in generator and discriminator
- **GAN training** with adversarial + reconstruction + perceptual + diversity (+ optional class) losses

### One-category assumption
This notebook is configured to train only on **`balloon_sketch`** paired data.

In [ ]:
# If needed (first run only):
# !pip install -q torch torchvision pillow tqdm matplotlib opencv-python scikit-image

import os
import random
from dataclasses import dataclass
from pathlib import Path
from typing import List, Tuple, Dict

import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

In [ ]:
@dataclass
class CFG:
    # --- data ---
    root = Path('./data/balloon_sketch')
    images_dir = root / 'images'      # RGB targets (preferred folder name)
    photos_dir = root / 'photos'      # optional backward-compatible folder name
    sketches_dir = root / 'sketches'  # hand-drawn sketches (5 sketches per image)
    hed_dir = root / 'hed_edges'      # generated edge sketches from photos

    # expected pairing setup
    sketches_per_image = 5

    image_size = 256
    batch_size = 8
    num_workers = 2
    val_ratio = 0.1

    # --- optimization ---
    epochs = 50
    lr_g = 2e-4
    lr_d = 2e-4
    beta1 = 0.5
    beta2 = 0.999

    # --- noise / latent ---
    z_dim = 128

    # --- loss weights (paper-inspired multi-loss setup) ---
    lambda_l1 = 100.0
    lambda_perc = 10.0
    lambda_div = 1.0
    lambda_cls = 0.1

    # one category only, but keep interface compatible
    num_classes = 1

cfg = CFG()
cfg.root.mkdir(parents=True, exist_ok=True)
print(cfg)

## 1) Prepare paired data for single class `balloon_sketch`

Expected directory layout (your setup):

```text
data/balloon_sketch/
  images/
    0001.jpg
    0002.jpg
    ...
  sketches/
    0001_1.png
    0001_2.png
    0001_3.png
    0001_4.png
    0001_5.png
    0002_1.png
    ...
```

- Each image can have multiple sketches (e.g., **5 sketches per image**).
- Matching supports exact stem match and common suffix patterns such as `_1`, `_2`, ..., `_5`, `-1`, `sketch1`, `sketch_1`.
- The train/val split is done by **image ID**, so sketches of the same image do not leak across splits.
- If hand-drawn sketches are missing, you can still train using generated HED edge maps from `images/` as sketch inputs.

In [ ]:
# HED-style edge augmentation step (paper data augmentation idea)
# Tries OpenCV Structured Edge (if available), otherwise falls back to Canny + morphology.

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.webp'}


def get_images_dir(cfg: CFG) -> Path:
    if cfg.images_dir.exists():
        return cfg.images_dir
    if cfg.photos_dir.exists():
        return cfg.photos_dir
    return cfg.images_dir


def _resize_keep(img: np.ndarray, size: int) -> np.ndarray:
    return cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)


def hed_like_edge_from_photo(photo_bgr: np.ndarray) -> np.ndarray:
    gray = cv2.cvtColor(photo_bgr, cv2.COLOR_BGR2GRAY)

    # fallback edge path that works everywhere
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blur, threshold1=60, threshold2=150)

    # Make line map sketch-like: clean + thin/consistent strokes
    kernel = np.ones((2, 2), np.uint8)
    edges = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel, iterations=1)

    # invert: white background + dark strokes (sketch convention)
    edge_inv = 255 - edges
    return edge_inv


def generate_hed_edges(cfg: CFG, overwrite: bool = False):
    cfg.hed_dir.mkdir(parents=True, exist_ok=True)
    images_dir = get_images_dir(cfg)
    photo_paths = sorted([p for p in images_dir.glob('*') if p.suffix.lower() in IMG_EXTS])

    if not photo_paths:
        raise FileNotFoundError(f'No images found in {images_dir}')

    for p in tqdm(photo_paths, desc='Generating HED-like edges'):
        out = cfg.hed_dir / f'{p.stem}.png'
        if out.exists() and not overwrite:
            continue
        img = cv2.imread(str(p))
        if img is None:
            continue
        edge = hed_like_edge_from_photo(img)
        edge = _resize_keep(edge, cfg.image_size)
        cv2.imwrite(str(out), edge)

    print('HED-like edge generation done:', cfg.hed_dir)

# Run once when images are available
# generate_hed_edges(cfg, overwrite=False)

In [ ]:
# Optional: use true HED model if you have OpenCV contrib + HED model file.
# Put model file at: data/balloon_sketch/hed_model/model.yml.gz

cfg_hed_model = cfg.root / 'hed_model' / 'model.yml.gz'


def try_true_hed(photo_bgr: np.ndarray, model_path: Path):
    if not model_path.exists():
        return None
    if not hasattr(cv2, 'ximgproc') or not hasattr(cv2.ximgproc, 'createStructuredEdgeDetection'):
        return None
    try:
        sed = cv2.ximgproc.createStructuredEdgeDetection(str(model_path))
        rgb = cv2.cvtColor(photo_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        e = sed.detectEdges(rgb)  # float [0,1]
        e = (e * 255).clip(0, 255).astype(np.uint8)
        e = 255 - e
        return e
    except Exception as ex:
        print('Falling back to canny-like edge due to:', ex)
        return None


def generate_edges_with_optional_true_hed(cfg: CFG, overwrite: bool = False):
    cfg.hed_dir.mkdir(parents=True, exist_ok=True)
    images_dir = get_images_dir(cfg)
    photo_paths = sorted([p for p in images_dir.glob('*') if p.suffix.lower() in IMG_EXTS])

    for p in tqdm(photo_paths, desc='Generating edge maps'):
        out = cfg.hed_dir / f'{p.stem}.png'
        if out.exists() and not overwrite:
            continue

        img = cv2.imread(str(p))
        if img is None:
            continue

        edge = try_true_hed(img, cfg_hed_model)
        if edge is None:
            edge = hed_like_edge_from_photo(img)

        edge = _resize_keep(edge, cfg.image_size)
        cv2.imwrite(str(out), edge)

    print('Edge-map generation done.')

# Preferred generation call (uses true HED if available, else fallback)
# generate_edges_with_optional_true_hed(cfg, overwrite=False)

In [ ]:
import re
from collections import defaultdict


def infer_image_id_from_sketch_stem(sketch_stem: str, image_stems: set) -> str:
    """
    Support 1-image-to-many-sketch naming patterns.
    Tries exact match first, then strips common trailing sketch indices.
    """
    if sketch_stem in image_stems:
        return sketch_stem

    candidates = [
        re.sub(r'([_-](?:sketch)?\d+)$', '', sketch_stem, flags=re.IGNORECASE),
        re.sub(r'((?:sketch)[_-]?\d+)$', '', sketch_stem, flags=re.IGNORECASE),
        re.sub(r'([_-]\d+)$', '', sketch_stem, flags=re.IGNORECASE),
    ]

    for c in candidates:
        if c in image_stems:
            return c

    return ''


def list_pairs(cfg: CFG) -> List[Tuple[Path, Path, str]]:
    images_dir = get_images_dir(cfg)
    photo_paths = sorted([p for p in images_dir.glob('*') if p.suffix.lower() in IMG_EXTS])
    photo_map = {p.stem: p for p in photo_paths}

    if not photo_map:
        raise FileNotFoundError(f'No images found in {images_dir}')

    # Prefer human sketches; fall back to generated edges
    sketch_src = cfg.sketches_dir if cfg.sketches_dir.exists() else cfg.hed_dir
    if not sketch_src.exists():
        raise FileNotFoundError(f'No sketch source found. Provide {cfg.sketches_dir} or generate {cfg.hed_dir}.')

    sketch_paths = sorted([p for p in sketch_src.glob('*') if p.suffix.lower() in IMG_EXTS])

    pairs = []
    skipped = []
    image_stems = set(photo_map.keys())
    for sk in sketch_paths:
        image_id = infer_image_id_from_sketch_stem(sk.stem, image_stems)
        if not image_id:
            skipped.append(sk.name)
            continue
        pairs.append((sk, photo_map[image_id], image_id))

    if not pairs:
        raise RuntimeError('No paired files found. Check sketch naming vs image naming.')

    # Report sketches-per-image statistics
    cnt = defaultdict(int)
    for _, _, image_id in pairs:
        cnt[image_id] += 1
    vals = sorted(cnt.values())
    print(f'Paired examples: {len(pairs)} | unique images: {len(cnt)} | sketch source: {sketch_src}')
    print(f'Sketches per image -> min:{vals[0]} median:{vals[len(vals)//2]} max:{vals[-1]} | expected ~{cfg.sketches_per_image}')
    if skipped:
        print(f'Unmatched sketches skipped: {len(skipped)} (showing first 10): {skipped[:10]}')

    return pairs


def train_val_split_by_image(pairs: List[Tuple[Path, Path, str]], val_ratio: float = 0.1, seed: int = 42):
    """Split by image_id to avoid leakage across 5 sketches of the same image."""
    rng = random.Random(seed)

    by_img = defaultdict(list)
    for item in pairs:
        by_img[item[2]].append(item)

    image_ids = list(by_img.keys())
    rng.shuffle(image_ids)

    n_val_imgs = max(1, int(len(image_ids) * val_ratio)) if len(image_ids) > 1 else 1
    val_ids = set(image_ids[:n_val_imgs])

    train, val = [], []
    for image_id, group in by_img.items():
        if image_id in val_ids:
            val.extend(group)
        else:
            train.extend(group)

    # if only one image exists, keep at least one train sample
    if len(train) == 0 and len(val) > 1:
        train.append(val.pop())

    return train, val

In [ ]:
class BalloonSketchDataset(Dataset):
    def __init__(self, pairs: List[Tuple[Path, Path, str]], image_size: int = 256):
        self.pairs = pairs
        self.t_img = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
        ])
        self.t_sketch = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.Grayscale(num_output_channels=1),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),
        ])

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, i):
        sketch_p, photo_p, image_id = self.pairs[i]
        sketch = Image.open(sketch_p).convert('RGB')
        photo = Image.open(photo_p).convert('RGB')

        sketch = self.t_sketch(sketch)
        photo = self.t_img(photo)

        # single category => class index always 0
        label = torch.tensor(0, dtype=torch.long)

        return {
            'sketch': sketch,
            'photo': photo,
            'label': label,
            'id': sketch_p.stem,
            'image_id': image_id,
        }


pairs = list_pairs(cfg)
train_pairs, val_pairs = train_val_split_by_image(pairs, cfg.val_ratio, SEED)

train_ds = BalloonSketchDataset(train_pairs, cfg.image_size)
val_ds = BalloonSketchDataset(val_pairs, cfg.image_size)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)

print('train pairs:', len(train_ds), 'val pairs:', len(val_ds))

In [ ]:
def denorm_img(x):
    return (x * 0.5 + 0.5).clamp(0, 1)

batch = next(iter(train_loader))
sk, ph = batch['sketch'][:4], batch['photo'][:4]

fig, ax = plt.subplots(4, 2, figsize=(6, 10))
for i in range(4):
    ax[i, 0].imshow(denorm_img(sk[i]).squeeze(0).cpu(), cmap='gray')
    ax[i, 0].set_title(f'Sketch {i}')
    ax[i, 0].axis('off')

    ax[i, 1].imshow(denorm_img(ph[i]).permute(1, 2, 0).cpu())
    ax[i, 1].set_title(f'Photo {i}')
    ax[i, 1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# --- MRU blocks (paper-inspired) ---

class MRUBlock(nn.Module):
    """
    Simplified Masked Residual Unit:
    - mixes current feature x with resized sketch input s
    - learns update gate and reset-like mask
    - residual output keeps optimization stable
    """
    def __init__(self, in_ch, out_ch, sketch_ch=1):
        super().__init__()
        self.in_proj = nn.Conv2d(in_ch, out_ch, 1)

        self.mask_conv = nn.Sequential(
            nn.Conv2d(in_ch + sketch_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.Sigmoid(),
        )
        self.update_conv = nn.Sequential(
            nn.Conv2d(in_ch + sketch_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.Sigmoid(),
        )
        self.candidate = nn.Sequential(
            nn.Conv2d(in_ch + sketch_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x, sketch):
        s = F.interpolate(sketch, size=x.shape[-2:], mode='bilinear', align_corners=False)
        xs = torch.cat([x, s], dim=1)

        m = self.mask_conv(xs)
        z = self.update_conv(xs)

        x_proj = self.in_proj(x)
        cand = self.candidate(torch.cat([x * m.mean(dim=1, keepdim=True), s], dim=1))

        out = (1 - z) * x_proj + z * cand
        return out


class MRUDown(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.mru = MRUBlock(in_ch, out_ch)
        self.down = nn.Conv2d(out_ch, out_ch, 4, stride=2, padding=1)

    def forward(self, x, s):
        x = self.mru(x, s)
        x = self.down(x)
        # Store skip after downsampling so decoder skip shapes match.
        x_skip = x
        return x, x_skip


class MRUUp(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, 4, stride=2, padding=1)
        self.mru = MRUBlock(out_ch, out_ch)

    def forward(self, x, s, skip=None):
        x = self.up(x)
        if skip is not None:
            x = x + skip
        x = self.mru(x, s)
        return x


class SketchyGenerator(nn.Module):
    def __init__(self, z_dim=128, base=64):
        super().__init__()
        self.stem = nn.Conv2d(1, base, 3, padding=1)

        self.d1 = MRUDown(base, base)        # 256 -> 128
        self.d2 = MRUDown(base, base * 2)    # 128 -> 64
        self.d3 = MRUDown(base * 2, base * 4)# 64 -> 32
        self.d4 = MRUDown(base * 4, base * 8)# 32 -> 16

        self.z_proj = nn.Sequential(
            nn.Linear(z_dim, base * 8 * 16 * 16),
            nn.ReLU(inplace=True),
        )

        self.u1 = MRUUp(base * 8, base * 4)  # 16 -> 32
        self.u2 = MRUUp(base * 4, base * 2)  # 32 -> 64
        self.u3 = MRUUp(base * 2, base)      # 64 -> 128
        self.u4 = MRUUp(base, base)          # 128 -> 256

        self.to_rgb = nn.Sequential(
            nn.Conv2d(base, 3, 3, padding=1),
            nn.Tanh(),
        )

    def forward(self, sketch, z):
        x = self.stem(sketch)
        x, s1 = self.d1(x, sketch)
        x, s2 = self.d2(x, sketch)
        x, s3 = self.d3(x, sketch)
        x, s4 = self.d4(x, sketch)

        bz = z.shape[0]
        zf = self.z_proj(z).view(bz, -1, 16, 16)
        x = x + zf

        x = self.u1(x, sketch, s3)
        x = self.u2(x, sketch, s2)
        x = self.u3(x, sketch, s1)
        x = self.u4(x, sketch, None)

        return self.to_rgb(x)


class SketchyDiscriminator(nn.Module):
    def __init__(self, base=64, num_classes=1):
        super().__init__()
        in_ch = 1 + 3
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, base, 4, 2, 1), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(base, base * 2, 4, 2, 1), nn.BatchNorm2d(base * 2), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(base * 2, base * 4, 4, 2, 1), nn.BatchNorm2d(base * 4), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(base * 4, base * 8, 4, 2, 1), nn.BatchNorm2d(base * 8), nn.LeakyReLU(0.2, inplace=True),
        )
        self.adv_head = nn.Conv2d(base * 8, 1, 3, padding=1)
        self.cls_head = nn.Linear(base * 8, max(1, num_classes))

    def forward(self, sketch, img):
        x = torch.cat([sketch, img], dim=1)
        f = self.net(x)
        adv = self.adv_head(f)  # patch logits
        pooled = F.adaptive_avg_pool2d(f, 1).flatten(1)
        cls = self.cls_head(pooled)
        return adv, cls


G = SketchyGenerator(cfg.z_dim).to(device)
D = SketchyDiscriminator(num_classes=cfg.num_classes).to(device)

print('G params:', sum(p.numel() for p in G.parameters()) / 1e6, 'M')
print('D params:', sum(p.numel() for p in D.parameters()) / 1e6, 'M')

In [ ]:
# --- Losses ---

def d_hinge_loss(real_logits, fake_logits):
    return torch.relu(1.0 - real_logits).mean() + torch.relu(1.0 + fake_logits).mean()


def g_hinge_loss(fake_logits):
    return -fake_logits.mean()


class VGGPerceptual(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_FEATURES).features[:16]
        for p in vgg.parameters():
            p.requires_grad = False
        self.vgg = vgg.eval()

    def forward(self, x):
        # x in [-1,1] -> [0,1]
        x = (x + 1.0) / 2.0
        mean = torch.tensor([0.485, 0.456, 0.406], device=x.device).view(1, 3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225], device=x.device).view(1, 3, 1, 1)
        x = (x - mean) / std
        return self.vgg(x)


perc_net = VGGPerceptual().to(device)

opt_g = torch.optim.Adam(G.parameters(), lr=cfg.lr_g, betas=(cfg.beta1, cfg.beta2))
opt_d = torch.optim.Adam(D.parameters(), lr=cfg.lr_d, betas=(cfg.beta1, cfg.beta2))

cls_criterion = nn.CrossEntropyLoss()


def diversity_loss(fake1, fake2, z1, z2, eps=1e-6):
    # encourages output change with latent change
    img_dist = (fake1 - fake2).abs().mean(dim=[1, 2, 3])
    z_dist = (z1 - z2).abs().mean(dim=1)
    ratio = img_dist / (z_dist + eps)
    return -ratio.mean()

In [ ]:
def train_one_epoch(epoch: int):
    G.train()
    D.train()

    log = {'d': 0.0, 'g': 0.0, 'adv': 0.0, 'l1': 0.0, 'perc': 0.0, 'div': 0.0, 'cls': 0.0}
    n = 0

    for batch in tqdm(train_loader, desc=f'Train {epoch:03d}'):
        sketch = batch['sketch'].to(device, non_blocking=True)
        real = batch['photo'].to(device, non_blocking=True)
        label = batch['label'].to(device, non_blocking=True)
        bsz = sketch.size(0)

        # ---- D step ----
        z = torch.randn(bsz, cfg.z_dim, device=device)
        with torch.no_grad():
            fake = G(sketch, z)

        real_logits, real_cls = D(sketch, real)
        fake_logits, _ = D(sketch, fake)

        loss_d_adv = d_hinge_loss(real_logits, fake_logits)
        if cfg.num_classes > 1:
            loss_d_cls = cls_criterion(real_cls, label)
        else:
            loss_d_cls = torch.tensor(0.0, device=device)

        loss_d = loss_d_adv + cfg.lambda_cls * loss_d_cls

        opt_d.zero_grad(set_to_none=True)
        loss_d.backward()
        opt_d.step()

        # ---- G step ----
        z1 = torch.randn(bsz, cfg.z_dim, device=device)
        z2 = torch.randn(bsz, cfg.z_dim, device=device)

        fake1 = G(sketch, z1)
        fake2 = G(sketch, z2)

        fake_logits, fake_cls = D(sketch, fake1)

        loss_g_adv = g_hinge_loss(fake_logits)
        loss_l1 = F.l1_loss(fake1, real)

        with torch.no_grad():
            real_f = perc_net(real)
        fake_f = perc_net(fake1)
        loss_perc = F.l1_loss(fake_f, real_f)

        loss_div = diversity_loss(fake1, fake2, z1, z2)

        if cfg.num_classes > 1:
            loss_g_cls = cls_criterion(fake_cls, label)
        else:
            loss_g_cls = torch.tensor(0.0, device=device)

        loss_g = (
            loss_g_adv
            + cfg.lambda_l1 * loss_l1
            + cfg.lambda_perc * loss_perc
            + cfg.lambda_div * loss_div
            + cfg.lambda_cls * loss_g_cls
        )

        opt_g.zero_grad(set_to_none=True)
        loss_g.backward()
        opt_g.step()

        n += 1
        log['d'] += loss_d.item()
        log['g'] += loss_g.item()
        log['adv'] += loss_g_adv.item()
        log['l1'] += loss_l1.item()
        log['perc'] += loss_perc.item()
        log['div'] += loss_div.item()
        log['cls'] += loss_g_cls.item()

    for k in log:
        log[k] /= max(1, n)
    return log


@torch.no_grad()
def preview_samples(n_show=4):
    G.eval()
    batch = next(iter(val_loader))
    sketch = batch['sketch'][:n_show].to(device)
    real = batch['photo'][:n_show].to(device)
    z = torch.randn(sketch.size(0), cfg.z_dim, device=device)
    fake = G(sketch, z)

    fig, ax = plt.subplots(n_show, 3, figsize=(9, 3 * n_show))
    for i in range(n_show):
        ax[i, 0].imshow(denorm_img(sketch[i]).squeeze(0).cpu(), cmap='gray')
        ax[i, 0].set_title('Sketch')
        ax[i, 0].axis('off')

        ax[i, 1].imshow(denorm_img(fake[i]).permute(1, 2, 0).cpu())
        ax[i, 1].set_title('Generated')
        ax[i, 1].axis('off')

        ax[i, 2].imshow(denorm_img(real[i]).permute(1, 2, 0).cpu())
        ax[i, 2].set_title('Real')
        ax[i, 2].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
# --- Main training loop ---

save_dir = Path('./checkpoints_balloon')
save_dir.mkdir(exist_ok=True, parents=True)

history = []
for epoch in range(1, cfg.epochs + 1):
    logs = train_one_epoch(epoch)
    history.append(logs)

    print(
        f"Epoch {epoch:03d} | "
        f"D {logs['d']:.3f} | G {logs['g']:.3f} | "
        f"adv {logs['adv']:.3f} | l1 {logs['l1']:.3f} | "
        f"perc {logs['perc']:.3f} | div {logs['div']:.3f}"
    )

    if epoch % 5 == 0 or epoch == 1:
        preview_samples(n_show=min(4, cfg.batch_size))
        ckpt = {
            'G': G.state_dict(),
            'D': D.state_dict(),
            'cfg': cfg.__dict__,
            'epoch': epoch,
            'history': history,
        }
        torch.save(ckpt, save_dir / f'sketchygan_balloon_e{epoch:03d}.pt')

print('Training completed.')

In [ ]:
# --- Inference on a single sketch file ---

@torch.no_grad()
def load_generator(ckpt_path: str):
    ckpt = torch.load(ckpt_path, map_location=device)
    gen = SketchyGenerator(cfg.z_dim).to(device)
    gen.load_state_dict(ckpt['G'])
    gen.eval()
    return gen


def preprocess_sketch(path: str, image_size: int = 256):
    t = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.Grayscale(num_output_channels=1),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5]),
    ])
    x = Image.open(path).convert('RGB')
    return t(x).unsqueeze(0)


@torch.no_grad()
def generate_from_sketch(gen, sketch_path: str, n_samples: int = 4):
    sk = preprocess_sketch(sketch_path, cfg.image_size).to(device)
    fig, ax = plt.subplots(1, n_samples + 1, figsize=(3 * (n_samples + 1), 3))

    ax[0].imshow(denorm_img(sk[0]).squeeze(0).cpu(), cmap='gray')
    ax[0].set_title('Input sketch')
    ax[0].axis('off')

    for i in range(n_samples):
        z = torch.randn(1, cfg.z_dim, device=device)
        out = gen(sk, z)
        ax[i + 1].imshow(denorm_img(out[0]).permute(1, 2, 0).cpu())
        ax[i + 1].set_title(f'Sample {i + 1}')
        ax[i + 1].axis('off')

    plt.tight_layout()
    plt.show()

# Example:
# gen = load_generator('./checkpoints_balloon/sketchygan_balloon_e050.pt')
# generate_from_sketch(gen, './data/balloon_sketch/sketches/0001_1.png', n_samples=4)

## Notes for your one-category run (`balloon_sketch`)

- Keep images in `data/balloon_sketch/images` and sketches in `data/balloon_sketch/sketches`.
- Your many-to-one setup is supported: **5 sketches for each image**.
- Recommended naming example:
  - image: `0001.jpg`
  - sketches: `0001_1.png`, `0001_2.png`, `0001_3.png`, `0001_4.png`, `0001_5.png`
- Matching also handles common variants (`0001-1`, `0001_sketch1`, `0001sketch1`).
- Train/val split is by image ID, so all sketches from one image stay in the same split.
- If you only have photos, run edge generation first and train with `hed_edges` as sketch input.

This keeps the paper's **HED + MRU + GAN** flow while fitting your single-category, multi-sketch dataset format.

In [ ]:
# --- Inception Score (paper-style: Inception-v3 softmax + 10-split KL) ---

@torch.no_grad()
def sample_generated_images(gen, loader, num_images: int = 500):
    """
    Generate a fixed number of fake images from sketches.
    For one-category datasets, this still follows the standard IS protocol.
    """
    gen.eval()
    out = []
    total = 0

    for batch in loader:
        sketch = batch['sketch'].to(device)
        bsz = sketch.size(0)
        z = torch.randn(bsz, cfg.z_dim, device=device)
        fake = gen(sketch, z).detach().cpu()

        out.append(fake)
        total += fake.size(0)
        if total >= num_images:
            break

    if not out:
        raise RuntimeError('No generated images collected. Check your dataloader.')

    return torch.cat(out, dim=0)[:num_images]


@torch.no_grad()
def inception_score(
    images: torch.Tensor,
    device: torch.device,
    batch_size: int = 32,
    splits: int = 10,
):
    """
    Compute Inception Score exactly in the common GAN-paper style:
      IS = exp( E_x [ KL( p(y|x) || p(y) ) ] )
    reported as mean +- std over N splits.

    images: tensor [N,3,H,W] in [-1,1] or [0,1]
    """
    assert images.ndim == 4 and images.size(1) == 3, 'Expected images of shape [N,3,H,W]'

    n = images.size(0)
    if n < 2:
        raise ValueError('Need at least 2 images to compute Inception Score.')

    splits = min(splits, n)

    # Torchvision compatibility: some versions require aux_logits=True when pretrained weights are used.
    try:
        inception = models.inception_v3(
            weights=models.Inception_V3_Weights.IMAGENET1K_V1,
            aux_logits=True,
        )
    except (TypeError, ValueError):
        inception = models.inception_v3(weights=models.Inception_V3_Weights.IMAGENET1K_V1)

    inception = inception.to(device)
    inception.eval()

    preds = []
    for i in range(0, n, batch_size):
        x = images[i:i + batch_size].to(device)

        # Normalize to [0,1] if currently in [-1,1]
        if x.min().item() < 0:
            x = (x + 1.0) / 2.0
        x = x.clamp(0, 1)

        x = F.interpolate(x, size=(299, 299), mode='bilinear', align_corners=False)
        logits = inception(x)
        # Handle return type differences across torchvision versions.
        if hasattr(logits, 'logits'):
            logits = logits.logits
        elif isinstance(logits, (tuple, list)):
            logits = logits[0]

        probs = F.softmax(logits, dim=1)
        preds.append(probs.cpu())

    preds = torch.cat(preds, dim=0).numpy()

    split_scores = []
    for k in range(splits):
        start = k * n // splits
        end = (k + 1) * n // splits
        part = preds[start:end]
        if len(part) == 0:
            continue

        py = np.mean(part, axis=0, keepdims=True)
        kl = part * (np.log(part + 1e-16) - np.log(py + 1e-16))
        kl = np.sum(kl, axis=1)
        split_scores.append(np.exp(np.mean(kl)))

    return float(np.mean(split_scores)), float(np.std(split_scores))


# Example usage after training:
# fake_images = sample_generated_images(G, val_loader, num_images=500)
# is_mean, is_std = inception_score(fake_images, device=device, batch_size=32, splits=10)
# print(f'Inception Score (10 splits): {is_mean:.3f} ± {is_std:.3f}')

# Optional: evaluate from a saved checkpoint
# gen = load_generator('./checkpoints_balloon/sketchygan_balloon_e100.pt')
# fake_images = sample_generated_images(gen, val_loader, num_images=500)
# is_mean, is_std = inception_score(fake_images, device=device)
# print(f'Inception Score: {is_mean:.3f} ± {is_std:.3f}')

In [ ]:
# --- FID (Fréchet Inception Distance) for single-category evaluation ---
# Lower is better. This metric is usually more meaningful than IS for one-category generation.


def _build_inception_for_fid(device: torch.device):
    # Torchvision compatibility across versions.
    try:
        model = models.inception_v3(
            weights=models.Inception_V3_Weights.IMAGENET1K_V1,
            aux_logits=True,
        )
    except (TypeError, ValueError):
        model = models.inception_v3(weights=models.Inception_V3_Weights.IMAGENET1K_V1)

    model = model.to(device).eval()
    return model


@torch.no_grad()
def _inception_pool3_features(model, x: torch.Tensor) -> torch.Tensor:
    """Return 2048-d pool3 features from Inception-v3."""
    # x: [N,3,299,299] in [0,1]
    x = model.Conv2d_1a_3x3(x)
    x = model.Conv2d_2a_3x3(x)
    x = model.Conv2d_2b_3x3(x)
    x = model.maxpool1(x)
    x = model.Conv2d_3b_1x1(x)
    x = model.Conv2d_4a_3x3(x)
    x = model.maxpool2(x)
    x = model.Mixed_5b(x)
    x = model.Mixed_5c(x)
    x = model.Mixed_5d(x)
    x = model.Mixed_6a(x)
    x = model.Mixed_6b(x)
    x = model.Mixed_6c(x)
    x = model.Mixed_6d(x)
    x = model.Mixed_6e(x)
    x = model.Mixed_7a(x)
    x = model.Mixed_7b(x)
    x = model.Mixed_7c(x)
    x = model.avgpool(x)
    x = torch.flatten(x, 1)  # [N, 2048]
    return x


@torch.no_grad()
def _prep_for_inception(x: torch.Tensor) -> torch.Tensor:
    # Convert from [-1,1] to [0,1] if needed.
    if x.min().item() < 0:
        x = (x + 1.0) / 2.0
    x = x.clamp(0, 1)
    x = F.interpolate(x, size=(299, 299), mode='bilinear', align_corners=False)
    return x


@torch.no_grad()
def collect_real_fake_features_for_fid(gen, loader, device, num_images: int = 500):
    model = _build_inception_for_fid(device)
    gen.eval()

    real_feats = []
    fake_feats = []
    total = 0

    for batch in loader:
        sketch = batch['sketch'].to(device)
        real = batch['photo'].to(device)

        bsz = sketch.size(0)
        z = torch.randn(bsz, cfg.z_dim, device=device)
        fake = gen(sketch, z)

        real_in = _prep_for_inception(real)
        fake_in = _prep_for_inception(fake)

        real_f = _inception_pool3_features(model, real_in)
        fake_f = _inception_pool3_features(model, fake_in)

        real_feats.append(real_f.cpu())
        fake_feats.append(fake_f.cpu())

        total += bsz
        if total >= num_images:
            break

    if not real_feats:
        raise RuntimeError('No features collected for FID. Check dataloader/content.')

    real_feats = torch.cat(real_feats, dim=0)[:num_images]
    fake_feats = torch.cat(fake_feats, dim=0)[:num_images]
    return real_feats, fake_feats


def _mean_and_cov(feats: torch.Tensor):
    feats = feats.double()
    mu = torch.mean(feats, dim=0)
    xc = feats - mu
    cov = (xc.T @ xc) / max(1, feats.shape[0] - 1)
    return mu, cov


def _sqrtm_psd(mat: torch.Tensor, eps: float = 1e-8):
    # Eigen-decomposition for positive semi-definite matrices.
    vals, vecs = torch.linalg.eigh(mat)
    vals = torch.clamp(vals, min=eps)
    sqrt_vals = torch.sqrt(vals)
    return vecs @ torch.diag(sqrt_vals) @ vecs.T


def frechet_distance_from_features(real_feats: torch.Tensor, fake_feats: torch.Tensor, eps: float = 1e-6) -> float:
    mu1, cov1 = _mean_and_cov(real_feats)
    mu2, cov2 = _mean_and_cov(fake_feats)

    d = mu1.shape[0]
    eye = torch.eye(d, dtype=torch.float64)
    cov1 = cov1 + eps * eye
    cov2 = cov2 + eps * eye

    diff = mu1 - mu2
    cov1_sqrt = _sqrtm_psd(cov1)
    middle = cov1_sqrt @ cov2 @ cov1_sqrt
    middle = (middle + middle.T) * 0.5
    middle_sqrt = _sqrtm_psd(middle)

    fid = diff.dot(diff) + torch.trace(cov1 + cov2 - 2.0 * middle_sqrt)
    return float(fid.item())


@torch.no_grad()
def compute_fid(gen, loader, device, num_images: int = 500):
    real_feats, fake_feats = collect_real_fake_features_for_fid(gen, loader, device, num_images=num_images)
    fid_value = frechet_distance_from_features(real_feats, fake_feats)
    return fid_value, real_feats.shape[0]


# Example usage (current in-memory generator):
# fid, n_used = compute_fid(G, val_loader, device=device, num_images=500)
# print(f'FID ({n_used} samples): {fid:.3f} (lower is better)')

# Example usage (from checkpoint):
# gen = load_generator('./checkpoints_balloon/sketchygan_balloon_e100.pt')
# fid, n_used = compute_fid(gen, val_loader, device=device, num_images=500)
# print(f'FID ({n_used} samples): {fid:.3f} (lower is better)')